# 09b — Reconstrucción reproducible del baseline

Este notebook reconstruye la extracción inicial conservando la configuración de junio: **gpt-4o-mini**, temperatura 0 y máximo 60 chunks. Lee los datos desde GitHub y descarga todas las salidas en un ZIP.

> La clave se introduce de forma privada y nunca se guarda en GitHub ni en los archivos de salida.

In [ ]:
!pip -q install pandas pyarrow openpyxl openai
!rm -rf /content/proyecto_ActividadGrado-Riesgos
!git clone -q --branch dia-01-diagnostico-falsos-positivos https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git /content/proyecto_ActividadGrado-Riesgos
print('Repositorio y dependencias preparados.')

## 1. Configurar la clave privada
Pega tu clave cuando aparezca el campo. Los caracteres no serán visibles.

In [ ]:
import os
from getpass import getpass
api_key = getpass('Ingrese OPENAI_API_KEY: ')
if not api_key.strip():
    raise ValueError('La clave no puede estar vacía.')
os.environ['OPENAI_API_KEY'] = api_key.strip()
del api_key
print('Clave cargada solo en la sesión actual.')

## 2. Ejecutar la extracción
La ejecución realiza 60 llamadas como máximo y guarda un checkpoint después de cada chunk.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path('/content/proyecto_ActividadGrado-Riesgos')
CHUNKS = REPO / 'data/processed/chunks/chunks_recursive.parquet'
OUTPUT = Path('/content/resultados_baseline')
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    'python', str(REPO / 'src/risk/extract_risks_baseline.py'),
    '--chunks', str(CHUNKS),
    '--output-dir', str(OUTPUT),
    '--max-chunks', '60',
]
subprocess.run(command, check=True, env=os.environ.copy())

## 3. Verificar los resultados
No es obligatorio obtener nuevamente 96 riesgos: esta es una nueva ejecución y debe conservarse tal como resulte.

In [ ]:
import json
import pandas as pd
from IPython.display import display

metadata = json.loads((OUTPUT / 'metadata_ejecucion.json').read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))
risks = pd.read_excel(OUTPUT / 'riesgos_evaluation_template.xlsx')
print('Candidatos extraídos:', len(risks))
display(risks.head(10))

## 4. Descargar todas las salidas
Adjunta el ZIP descargado en el chat para continuar con la evaluación manual y el diagnóstico del Día 1.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/resultados_baseline_riesgos', 'zip', OUTPUT)
files.download(zip_path)
print('Descarga preparada:', zip_path)